# Taxi Anomaly Detector - Environment Check

Run this notebook first to verify MinIO connectivity and that dummy taxi data was uploaded during chart installation.

Upstream taxi data can be sourced from **Zetaris** (installed in the cluster); this quickstart stores working copies in MinIO buckets `dev-data` and `data`.

In [ ]:
import os

print("MinIO endpoint:", os.getenv("AWS_S3_ENDPOINT"))
print("Default bucket:", os.getenv("AWS_S3_BUCKET"))
print("Dev bucket:", os.getenv("TAXI_DEV_DATA_BUCKET", "dev-data"))
print("Data object:", os.getenv("TAXI_DATA_OBJECT", "taxi_trips.csv"))

In [ ]:
import io

import boto3
import pandas as pd

endpoint = os.environ["AWS_S3_ENDPOINT"]
access_key = os.environ["AWS_ACCESS_KEY_ID"]
secret_key = os.environ["AWS_SECRET_ACCESS_KEY"]
object_key = os.getenv("TAXI_DATA_OBJECT", "taxi_trips.csv")
buckets = [
    os.getenv("TAXI_DEV_DATA_BUCKET", "dev-data"),
    os.getenv("TAXI_DATA_BUCKET", os.getenv("AWS_S3_BUCKET", "data")),
]

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

for bucket in buckets:
    print(f"\nChecking s3://{bucket}/")
    objects = s3.list_objects_v2(Bucket=bucket).get("Contents", [])
    if not objects:
        raise RuntimeError(f"No objects found in bucket '{bucket}'")

    keys = [obj["Key"] for obj in objects]
    print("Objects:", keys)

    if object_key not in keys:
        raise RuntimeError(f"Expected object '{object_key}' not found in bucket '{bucket}'")

    body = s3.get_object(Bucket=bucket, Key=object_key)["Body"].read()
    df = pd.read_csv(io.BytesIO(body))
    print(f"Loaded {len(df)} rows from s3://{bucket}/{object_key}")
    display(df.head())

print("\nAll checks passed.")